# BFCL-India v2 Training (Colab)

Run on T4 GPU. File → Open in VS Code with Colab extension.

In [1]:
!git clone https://github.com/bhavjeetsingh/bfcl-India.git
%cd bfcl-India
!pip install -q peft trl accelerate
!nvidia-smi

fatal: destination path 'bfcl-India' already exists and is not an empty directory.
/content/bfcl-India
/bin/bash: line 1: nvidia-smi: command not found


In [2]:
!pip install -q jsonschema python-dotenv tqdm
!python prepare_training_data.py --max-train 10000

[miss] glaive.jsonl not found — skipping glaive
[miss] xlam.jsonl not found — skipping xlam
[miss] xlam.jsonl not found — skipping xlam_unfiltered
[miss] apigen_mt.jsonl not found — skipping apigen
[load] bfcl: 10 valid records
[load] indian: 1778 valid records
[dedup] 1788 -> 1787 after dedup
[pick] 1787 examples after balance + cap
[format] 1783 examples after chat-format conversion

{
  "raw_per_source": {
    "bfcl": 10,
    "indian": 1778
  },
  "after_dedup": 1787,
  "final_per_source": {
    "indian_train": 1773,
    "bfcl_seeds": 10
  },
  "train_count": 1605,
  "val_count": 178,
  "max_train": 10000,
  "val_frac": 0.1,
  "seed": 42
}

Wrote: data/train.jsonl
Wrote: data/val.jsonl


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_SEQ = 2048

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
print("Model loaded OK")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded OK


: 

In [4]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

: 

: 

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={"train": "data/train.jsonl", "val": "data/val.jsonl"})
print(dataset)

In [ ]:
def formatting_prompts_func(examples):
    outputs = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        outputs.append(text)
    return {"text": outputs}

dataset = dataset.map(formatting_prompts_func, batched=True)
print("Sample:")
print(dataset["train"][0]["text"][:500])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        per_device_eval_batch_size=1,
        eval_accumulation_steps=4,
        gradient_accumulation_steps=4,
        warmup_steps=50,
        num_train_epochs=1,
        learning_rate=1e-4,
        fp16=True,
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        optim="adamw_8bit",
        seed=3407,
        report_to="none",
        gradient_checkpointing=True,
    ),
)

trainer_stats = trainer.train()
print("Training complete!")

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
train_loss = [(x["step"], x["loss"]) for x in log_history if "loss" in x]
eval_loss = [(x["step"], x["eval_loss"]) for x in log_history if "eval_loss" in x]

if train_loss and eval_loss:
    plt.figure(figsize=(10, 6))
    plt.plot([s for s, _ in train_loss], [l for _, l in train_loss], label="train_loss")
    plt.plot([s for s, _ in eval_loss], [l for _, l in eval_loss], label="eval_loss")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title("v2 Training: Train vs Eval Loss")
    plt.legend()
    plt.grid(True)
    plt.savefig("loss_curve_v2.png", dpi=150, bbox_inches="tight")
    plt.show()
    
    final_train = train_loss[-1][1]
    final_eval = eval_loss[-1][1]
    gap = final_eval - final_train
    print(f"Train loss: {final_train:.4f}")
    print(f"Eval loss:  {final_eval:.4f}")
    print(f"Gap:        {gap:.4f}")
    if gap > 0.2:
        print("WARNING: Overfitting.")
    else:
        print("OK: Acceptable generalization.")
else:
    print("No eval loss recorded.")

In [ ]:
model.save_pretrained("v2_lora_adapter")
tokenizer.save_pretrained("v2_lora_adapter")
print("LoRA adapter saved locally")

In [ ]:
!nvidia-smi